In [ ]:
import os
import re
import json
import time

from pathlib import Path
from collections import defaultdict
from semanticher.onto import OntologyDAG
from semanticher.data import load_table_list, load_tables

from langchain_openai.chat_models.base import BaseChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# =========================
# Base folders
# =========================
REPO_ROOT       = Path().resolve().parents[1]
BASE_RESULT_DIR = REPO_ROOT / "results" / "main_msv_structured"
CANDIDATE_DIR   = BASE_RESULT_DIR / "Result_msv_structured" / "Candidate_msv_structured"
LOG_DIR         = BASE_RESULT_DIR / "Promptinput_msv_structured"
SUMMARY_DIR     = BASE_RESULT_DIR / "Tokenusage_Duration_msv_structured"

os.makedirs(CANDIDATE_DIR, exist_ok=True)
os.makedirs(LOG_DIR,       exist_ok=True)
os.makedirs(SUMMARY_DIR,   exist_ok=True)


# =========================
# Data / ontology
# =========================
dag = OntologyDAG()
dag.build_dag()

ontology_json = []
for level1_uri in dag.edges.get(dag.root, []):
    level1_name = dag.name_of_uri(level1_uri)
    level2 = [dag.name_of_uri(c_uri) for c_uri in dag.edges.get(level1_uri, [])]
    ontology_json.append({
        "level1": level1_name,
        "level2": level2
    })

for v in ontology_json:
    print(v["level1"])
    for c in v["level2"]:
        print("\t", c)

n_level1 = len(ontology_json)
n_level2 = sum(len(v["level2"]) for v in ontology_json)
print("Number of level 1:", n_level1)
print("Number of level 2:", n_level2)

df_table_list = load_table_list()
dict_tables = load_tables()

k = 5


def dataframe_to_markdown(df, k=5):
    subset = df.head(k)
    headers = "| " + " | ".join(subset.columns) + " |"
    sep = "| " + " | ".join(["---"] * len(subset.columns)) + " |"

    rows = []
    for _, row in subset.iterrows():
        row_str = "| " + " | ".join(map(str, row.values)) + " |"
        rows.append(row_str)

    return headers + "\n" + sep + "\n" + "\n".join(rows)


# =========================
# Model aliases / experiments
# =========================
MODEL_ALIAS = {
    "deepseek-chat": "dsk",
    "gemini-2.0-flash": "gem",
    "gpt-4.1-mini": "gpt",
}

EXPERIMENTS = [
    ["deepseek-chat"],
    ["gemini-2.0-flash"],
    ["gpt-4.1-mini"],
    ["deepseek-chat", "gemini-2.0-flash"],
    ["deepseek-chat", "gpt-4.1-mini"],
    ["gemini-2.0-flash", "gpt-4.1-mini"],
    ["deepseek-chat", "gemini-2.0-flash", "gpt-4.1-mini"],
]


def build_experiment_tag(model_names):
    aliases = [MODEL_ALIAS[m] for m in model_names]
    prefix = {1: "single", 2: "two", 3: "three"}[len(model_names)]
    return f"{prefix}_{'_'.join(aliases)}"


def build_candidate_path(model_names):
    tag = build_experiment_tag(model_names)
    return CANDIDATE_DIR / f"candidate_msv_structured_{tag}.json"

def build_logs_path(model_names):
    tag = build_experiment_tag(model_names)
    return LOG_DIR / f"msv_structured_{tag}_logs.json"

def build_summary_path(model_names):
    tag = build_experiment_tag(model_names)
    return SUMMARY_DIR / f"summary_msv_structured_{tag}.json"

In [ ]:
# =========================
# LLMs
# =========================
llm1 = BaseChatOpenAI(
    model="deepseek-chat",
    openai_api_key="",
    openai_api_base="https://api.deepseek.com",
    temperature=0,
    max_tokens=8192,
)

llm2 = BaseChatOpenAI(
    model="gemini-2.0-flash",
    openai_api_key="",
    openai_api_base="https://generativelanguage.googleapis.com/v1beta/openai",
    temperature=0,
    max_tokens=None,
)

llm3 = BaseChatOpenAI(
    model="gpt-4.1-mini",
    openai_api_key="",
    openai_api_base="https://api.openai.com/v1",
    temperature=0,
    max_tokens=None,
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a knowledgeable assistant who helps map tabular data columns to ontology classes. You have expert knowledge in semantic table annotation, i.e., table column-to-ontology class mappings. Your task is to map all columns of a given table to ontology classes based on a provided ontology DAG with strict hierarchical constraints. You must strictly select from the provided ontology DAG only."
        ),
        (
            "human",
            """
            We have a table named "{table_name}" which has several columns. Below is the table in Markdown format, including column headers and a few example rows:

            {table_in_markdown}

            Below is the ontology DAG provided as a JSON array.

            Each item represents one Level 1 class and has the following structure:
            - "level1": the name of a Level 1 class
            - "level2": a list of all valid Level 2 subclasses of that Level 1 class

            Ontology DAG (JSON):
            {ontology_dag}

            Hierarchy rules:
            1) You may ONLY use class names that appear in the ontology JSON.
            2) A Level 1 class must come from the "level1" field of an item.
            3) A Level 2 class must appear in the "level2" list of the corresponding Level 1 class.
            4) If you output a Level 2 class, you MUST also output its parent Level 1 class.
            5) Do NOT invent new classes or hierarchy relations.
            6) If the same Level 2 class appears under multiple valid Level 1 classes in the ontology DAG, and that Level 2 class is plausible for the column, output all plausible valid (Level1 → Level2) paths.

            Task:
            For each column, output multiple plausible ontology class paths (Top-k).

            Each path must be either:
            - Level 1 only, or
            - a valid (Level1 → Level2) hierarchy defined in the ontology JSON.

            Decision rules:
            - Analyze all columns jointly.
            - Use column names, sample values, data types, and units as evidence.
            - Prefer the most specific class supported by the column evidence.
            - Select a Level 2 subclass when it clearly fits the column semantics.
            - If a Level 2 subclass is not clearly supported, it is acceptable to output only the Level 1 class.
            - Avoid selecting overly generic Level 1 classes when a more specific Level 2 subclass is well supported by the evidence.
            - If no suitable class exists, return an empty paths list.
            - When selecting a Level 2 class, check whether it has multiple valid Level 1 parents in the ontology DAG.
            - If multiple parent paths are valid and plausible, output all of them rather than only one.

            Output format:
            - Output ONLY a valid JSON array.
            - Do NOT include any explanatory text.
            - Each item in the array must correspond to one input column, in the same order as the table.
            - Each item must have the following structure:

            {{
                "column_name": "<exact column header>",
                "paths": [
                    ["Level1", confidence],
                    ["Level1", "Level2", confidence]
                ]
            }}

            Path rules:
            - Each element in "paths" represents exactly one class path.
            - If a path contains only a Level 1 class, use:
              ["Level1", confidence]
            - If a path contains both Level 1 and Level 2 classes, use:
              ["Level1", "Level2", confidence]
            - The first class in each path is always Level 1.
            - The second class, if present, is always Level 2.
            - Confidence is the confidence of the whole path, not separate nodes.
            - Confidence values must be numbers between 0 and 1.
            - Order paths by descending confidence.
            - Do not merge or omit valid parent-child paths just because they share the same Level 2 class name.
            - If no suitable class exists for a column, return:
              {{"column_name": "<exact column header>", "paths": []}}

            Examples:

            [{{"column_name":"Date","paths":[["Property","Time",0.81],["TemporalEntity","Instant",0.80]]}}]

            [{{"column_name":"Bill Total per CPE - (€)","paths":[["ObservationProperty","EnergyParameter",0.85]]}}]

            [{{"column_name":"TOTAL MONTHLY CONSUMPTION GSL (M3) Diesel A","paths":[["Property","Energy",0.95]]}}]

            [{{"column_name":"VALUE","paths":[["ObservationValue",0.85]]}}]

            [{{"column_name":"idStation","paths":[]}}]

            [{{"column_name":"CONSUMER CENTER","paths":[["Building1",0.81],["PhysicalObject","Building1",0.85],["Space","Building1",0.85]]}}]

            Important:
            - You MUST choose only from the class names that appear in the ontology JSON.
            - Do NOT invent new class names.
            - Before producing the final answer, verify that every Level1 → Level2 pair exists in the ontology JSON.
            - Return ONLY the JSON array.
            """
        ),
    ]
)

CHAIN_REGISTRY = {
    "deepseek-chat": prompt | llm1,
    "gemini-2.0-flash": prompt | llm2,
    "gpt-4.1-mini": prompt | llm3,
}

In [ ]:
def extract_json_from_response(text):
    """
    Robust JSON extractor from LLM responses.
    """
    m = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.DOTALL)
    s = m.group(1).strip() if m else text.strip()

    if '\\"' in s:
        try:
            unescaped = json.loads(f'"{s}"')
            if isinstance(unescaped, str):
                s = unescaped.strip()
        except Exception:
            pass

    start = s.find("[")
    end = s.rfind("]")
    if start != -1 and end != -1 and end > start:
        return s[start:end + 1]

    start = s.find("{")
    end = s.rfind("}")
    if start != -1 and end != -1 and end > start:
        return s[start:end + 1]

    return s


def normalize_confidence(x):
    try:
        x = float(x)
    except Exception:
        return None
    if x < 0:
        x = 0.0
    if x > 1:
        x = 1.0
    return round(x, 4)


def process_llm_output(llm_output_str, table_id, table_name, table_columns):
    """
    Convert raw LLM JSON output to per-column path-level records.

    Output record format:
    {
        "table_id": ...,
        "table_name": ...,
        "column_name": ...,
        "column_id": ...,
        "class": [
            ["Level1", conf],
            ["Level1", "Level2", conf]
        ]
    }
    """
    results = []

    try:
        parsed = json.loads(llm_output_str)
        if not isinstance(parsed, list):
            parsed = []
    except json.JSONDecodeError as e:
        print(f"JSON fail on {table_id}: {e}")
        parsed = []

    parsed_by_index = {}
    for i, col in enumerate(parsed):
        if isinstance(col, dict):
            parsed_by_index[i] = col

    for i, column_name in enumerate(table_columns):
        col = parsed_by_index.get(i, {})
        raw_paths = col.get("paths", [])

        best_scores = {}

        for p in raw_paths:
            if not isinstance(p, list):
                continue

            # Level 1 only
            if len(p) == 2:
                level1, conf = p
                if not isinstance(level1, str) or level1 is None:
                    continue
                conf = normalize_confidence(conf)
                if conf is None:
                    continue
                key = (level1,)
                best_scores[key] = max(best_scores.get(key, 0), conf)

            # Level 1 -> Level 2
            elif len(p) == 3:
                level1, level2, conf = p
                if not isinstance(level1, str) or level1 is None:
                    continue
                if not isinstance(level2, str) or level2 is None:
                    continue
                conf = normalize_confidence(conf)
                if conf is None:
                    continue
                key = (level1, level2)
                best_scores[key] = max(best_scores.get(key, 0), conf)

        normalized_paths = []
        for key, conf in best_scores.items():
            if len(key) == 1:
                normalized_paths.append([key[0], conf])
            else:
                normalized_paths.append([key[0], key[1], conf])

        normalized_paths.sort(key=lambda x: x[-1], reverse=True)

        results.append(
            {
                "table_id": table_id,
                "table_name": table_name,
                "column_name": column_name,
                "column_id": i,
                "class": normalized_paths,
            }
        )

    return results

In [ ]:
def run_single_experiment(model_names):
    chains = [CHAIN_REGISTRY[name] for name in model_names]
    num_agents = len(model_names)

    candidate_file_path = build_candidate_path(model_names)
    logs_path = build_logs_path(model_names)
    summary_path = build_summary_path(model_names)

    print("=" * 80)
    print(f"Running experiment: {build_experiment_tag(model_names)}")
    print("Models:", model_names)
    print("Candidate file:", candidate_file_path)
    print("Logs file:", logs_path)
    print("=" * 80)

    start_time = time.perf_counter()

    all_llm_records = []
    candidate_results = []

    for table_id in df_table_list["table_id"].unique():
        table = dict_tables[table_id]
        table_name = df_table_list[df_table_list["table_id"] == table_id]["table_name"].values[0]
        table_in_markdown = dataframe_to_markdown(table, k)
        table_columns = list(table.columns)

        print("+" * 70)
        print(table_id, table_name)

        input_dict = {
            "table_name": table_name,
            "table_in_markdown": table_in_markdown,
            "ontology_dag": ontology_json,
        }

        for agent_id in range(num_agents):
            model_name = model_names[agent_id]
            chain = chains[agent_id]

            prompt_template = chain.first
            formatted_prompt = prompt_template.format(**input_dict)

            if isinstance(formatted_prompt, str):
                prompt_text = formatted_prompt.strip()
            else:
                messages = formatted_prompt.to_messages()
                prompt_text = "\n".join([f"[{m.type.upper()}]\n{m.content}" for m in messages])

            response = chain.invoke(input_dict)
            output_text = extract_json_from_response(response.content)

            usage_raw = response.response_metadata.get("token_usage", {})
            usage_clean = {
                "input_tokens": usage_raw.get("prompt_tokens", 0),
                "output_tokens": usage_raw.get("completion_tokens", 0),
                "total_tokens": usage_raw.get("total_tokens", 0),
            }

            parsed_results = process_llm_output(
                llm_output_str=output_text,
                table_id=table_id,
                table_name=table_name,
                table_columns=table_columns,
            )

            for item in parsed_results:
                candidate_results.append(
                    {
                        "table_id": item["table_id"],
                        "table_name": item["table_name"],
                        "model": model_name,
                        "column_name": item["column_name"],
                        "column_id": item["column_id"],
                        "class": item["class"],
                    }
                )

            # logs: one record per LLM call (debug purpose)
            all_llm_records.append(
                {
                    "table_id": table_id,
                    "table_name": table_name,
                    "model": model_name,
                    "prompt": prompt_text,
                    "output": response.content,
                    "usage": usage_clean,
                }
            )

            print(f"  [{model_name}] done")

    end_time = time.perf_counter()
    elapsed_time = end_time - start_time
    elapsed_minutes = elapsed_time / 60

    token_summary = {
        "total": {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0},
        "by_model": defaultdict(lambda: {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}),
    }

    for rec in all_llm_records:
        model = rec.get("model", "unknown_model")
        usage = rec.get("usage", {})
        for key in ["input_tokens", "output_tokens", "total_tokens"]:
            value = usage.get(key, 0) or 0
            token_summary["total"][key] += value
            token_summary["by_model"][model][key] += value

    summary = {
        "experiment": build_experiment_tag(model_names),
        "models": model_names,
        "run_time": {
            "seconds": round(elapsed_time, 2),
            "minutes": round(elapsed_minutes, 2),
        },
        "token_usage_summary": {
            "total": token_summary["total"],
            "by_model": dict(token_summary["by_model"]),
        },
    }

    with open(candidate_file_path, "w", encoding="utf-8") as f:
        json.dump(candidate_results, f, ensure_ascii=False, indent=2)

    with open(logs_path, "w", encoding="utf-8") as f:
        json.dump(all_llm_records, f, ensure_ascii=False, indent=2)

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print(f"Saved candidate -> {candidate_file_path}")
    print(f"Saved logs      -> {logs_path}")
    print(f"Saved summary   -> {summary_path}")

In [ ]:
def run_all_experiments():
    for model_names in EXPERIMENTS:
        run_single_experiment(model_names=model_names)

run_all_experiments()